<a href="https://colab.research.google.com/github/prakashdeepika/SadTalker/blob/feature%2Fgithub-actions-video-pipeline/github-actions-video-pipeline/colab_wan_motion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experimental Wan image-to-video motion (Google Colab)

This separate experiment uses the official Wan 2.1 14B image-to-video model to generate a short motion clip from one still image and a text prompt. It then adds your audio as the soundtrack. **It does not synchronize mouth movements to the audio**; Wan is not an audio lip-sync model. SadTalker remains available separately for face/lip animation, but compositing that onto a generated moving scene is not automated here.

Select **Runtime → Change runtime type → GPU** before running. Free GPU access and memory are not guaranteed. This 14B model is large; CPU offloading may be slow or still run out of memory on a free T4. If that happens, this notebook cannot make the job fit every free runtime. The generated clip is only a few seconds. Motion, faces, identity, clothing, and background can change or look unrealistic; this is an experiment, not guaranteed movie-quality output.

You choose when to upload a file in the upload cell below. The files are sent to the Google Colab runtime for processing. The notebook does not put them into GitHub. Download the result before the Colab session ends. Only upload media you have permission to use.

In [1]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU is attached. Choose Runtime > Change runtime type > GPU, reconnect, then rerun.")
gpu = torch.cuda.get_device_properties(0)
print("GPU:", gpu.name)
print("Total VRAM: {:.1f} GiB".format(gpu.total_memory / (1024 ** 3)))
print("Wan 14B may still exceed the available VRAM or run very slowly with CPU offloading.")

GPU: Tesla T4
Total VRAM: 14.6 GiB
Wan 14B may still exceed the available VRAM or run very slowly with CPU offloading.


In [2]:
# Install current Diffusers support and FFmpeg in this temporary Colab runtime.
import subprocess
import sys

subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "ffmpeg"], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "--upgrade",
    "diffusers", "transformers", "accelerate", "imageio[ffmpeg]", "sentencepiece",
], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '--upgrade', 'diffusers', 'transformers', 'accelerate', 'imageio[ffmpeg]', 'sentencepiece'], returncode=0)

In [ ]:
# Upload one still image and one soundtrack (MP3/WAV/M4A/FLAC/OGG).
from google.colab import files
from pathlib import Path
import shutil

uploaded = files.upload()
images = [name for name in uploaded if Path(name).suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}]
audios = [name for name in uploaded if Path(name).suffix.lower() in {".mp3", ".wav", ".m4a", ".flac", ".ogg"}]
if len(images) != 1 or len(audios) != 1:
    raise ValueError("Upload exactly one image and one audio file.")

work = Path("/content/wan-work")
work.mkdir(parents=True, exist_ok=True)
image_path = work / Path(images[0]).name
audio_path = work / Path(audios[0]).name
shutil.move(images[0], image_path)
shutil.move(audios[0], audio_path)
print("Image:", image_path.name, "| soundtrack:", audio_path.name)

In [ ]:
# Load Wan 2.1 I2V 14B with CPU offloading to reduce peak GPU memory.
# Model weights are large and will be downloaded into the temporary Colab VM.
import torch
from diffusers import WanImageToVideoPipeline
from diffusers.utils import export_to_video
from PIL import Image

model_id = "Wan-AI/Wan2.1-I2V-14B-480P-Diffusers"
pipe = WanImageToVideoPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()
if hasattr(pipe.vae, "enable_tiling"):
    pipe.vae.enable_tiling()

image = Image.open(image_path).convert("RGB")
# Wan 480p I2V uses a landscape 832x480 canvas; preserve the input's full scene.
canvas = Image.new("RGB", (832, 480), (0, 0, 0))
image.thumbnail((832, 480), Image.Resampling.LANCZOS)
canvas.paste(image, ((832 - image.width) // 2, (480 - image.height) // 2))
print("Prepared image:", canvas.size)

In [ ]:
# Edit this prompt to describe the desired motion. Keep it physically plausible and simple.
prompt = (
    "A realistic cinematic wide shot of the same rural village scene. "
    "The man in the light shirt speaks emphatically, makes a few natural small hand gestures, "
    "then takes a slow step forward. The second man remains nearby and reacts subtly. "
    "Keep the camera steady, preserve the dry village setting, natural daylight, and realistic movement."
)
negative_prompt = "blurry, distorted face, extra fingers, deformed hands, duplicate people, sudden camera shake, text, subtitles"

with torch.inference_mode():
    result = pipe(
        image=canvas,
        prompt=prompt,
        negative_prompt=negative_prompt,
        height=480,
        width=832,
        num_frames=49,
        num_inference_steps=20,
        guidance_scale=5.0,
    )
frames = result.frames[0]
silent_video = work / "wan-motion-silent.mp4"
export_to_video(frames, str(silent_video), fps=16)
print("Generated", len(frames), "frames:", silent_video)

In [ ]:
# Add the uploaded audio as a soundtrack. This muxes audio; it does not lip-sync it.
import subprocess

output_video = work / "wan-motion-with-audio.mp4"
subprocess.run([
    "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
    "-i", str(silent_video), "-i", str(audio_path),
    "-map", "0:v:0", "-map", "1:a:0", "-c:v", "copy", "-c:a", "aac", "-b:a", "192k",
    "-shortest", "-movflags", "+faststart", str(output_video),
], check=True)
probe = subprocess.run([
    "ffprobe", "-v", "error", "-show_entries", "stream=codec_type,width,height",
    "-of", "default=noprint_wrappers=1", str(output_video),
], check=True, capture_output=True, text=True)
print(probe.stdout)
if "codec_type=video" not in probe.stdout or "codec_type=audio" not in probe.stdout:
    raise RuntimeError("Expected both video and audio streams in the final MP4.")

In [ ]:
from IPython.display import Video, display
display(Video(str(output_video), embed=True))
files.download(str(output_video))